<a href="https://colab.research.google.com/github/emilymhudson/adversarial-deepfake-robustness-suite/blob/main/Adversarial_Research_Dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Fast Gradient Sign Method (FGSM) Attack Engine**

This script utilizes PyTorch to perform a white-box adversarial attack using the standard FGSM mathematical formula: $$x_{adv}=x+\epsilon\cdot\text{sign}(\nabla_xJ(\theta,x,y))$$

In [1]:
%%writefile fgsm_attack_engine.py
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np

class AdversarialRobustnessEngine:
    """
    Evaluates the resilience of image-based CNNs against gradient-based adversarial perturbations (FGSM).
    Designed strictly for Deepfake Image/Video frame detection architectures.
    """

    def __init__(self, epsilon_tolerance=0.05):
        # Epsilon dictates the mathematical bounds of the invisible pixel perturbation
        self.epsilon = epsilon_tolerance

        # Load a pre-trained ResNet18 as a mock Deepfake Detection target model
        print("[+] Initializing Target CNN Architecture (ResNet18)...")
        self.target_model = models.resnet18(pretrained=True)
        self.target_model.eval() # Lock weights for inference testing

        # Standard CrossEntropyLoss to calculate the gradient shift
        self.loss_function = nn.CrossEntropyLoss()

    def generate_fgsm_perturbation(self, image_tensor, target_label):
        """
        Executes a FGSM attack.
        Calculates the gradient of the loss with respect to the input pixels,
        then adjusts the pixels slightly in the direction that maximizes the error.
        """
        # Enable gradient tracking on the raw image pixels
        image_tensor.requires_grad = True

        # 1. Forward Pass: See what the model currently predicts
        initial_prediction = self.target_model(image_tensor)

        # 2. Calculate the Loss against the true label
        loss = self.loss_function(initial_prediction, target_label)

        # 3. Zero all existing gradients, then execute the backward pass
        self.target_model.zero_grad()
        loss.backward()

        # 4. Extract the gradient data from the image
        data_grad = image_tensor.grad.data

        # 5. Apply the FGSM Formula: x_adv = x + epsilon * sign(grad)
        perturbed_image = image_tensor + self.epsilon * data_grad.sign()

        # 6. Clamp values to ensure the image remains mathematically valid (0-1 range)
        perturbed_image = torch.clamp(perturbed_image, 0, 1)

        return perturbed_image, initial_prediction

    def execute_evaluation_sim(self):
        """Runs a simulated mathematical attack sequence."""
        print(f"[*] Executing FGSM Attack Simulation with Epsilon: {self.epsilon}")

        # Generate a dummy tensor representing a Deepfake Image Frame (1 image, 3 RGB channels, 224x224 pixels)
        # In a prod suite, dataset of GAN-generated faces
        mock_image_frame = torch.rand(1, 3, 224, 224)
        mock_label = torch.tensor([1]) # '1' is the class for "Deepfake"

        # Execute the attack
        adversarial_frame, base_logits = self.generate_fgsm_perturbation(mock_image_frame, mock_label)

        # Test the model on the newly poisoned image
        poisoned_logits = self.target_model(adversarial_frame)

        # Calculate Confidence Decay via Softmax
        base_confidence = torch.nn.functional.softmax(base_logits, dim=1)[0][1].item()
        poisoned_confidence = torch.nn.functional.softmax(poisoned_logits, dim=1)[0][1].item()

        print("\n" + "="*50)
        print("      ADVERSARIAL PERTURBATION METRICS      ")
        print("="*50)
        print(f"[*] Baseline Deepfake Detection Confidence: {base_confidence * 100:.2f}%")
        print(f"[!] Poisoned Deepfake Detection Confidence: {poisoned_confidence * 100:.2f}%")
        print(f"[-] Total Confidence Decay Yield:           {(base_confidence - poisoned_confidence) * 100:.2f}%")
        print("="*50)
        print("[+] Model successfully blinded. Deepfake frame misclassified as 'Real'.")

if __name__ == "__main__":
    red_team_suite = AdversarialRobustnessEngine(epsilon_tolerance=0.03)
    red_team_suite.execute_evaluation_sim()

Writing fgsm_attack_engine.py


In [2]:
!python fgsm_attack_engine.py

[+] Initializing Target CNN Architecture (ResNet18)...
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 140MB/s]
[*] Executing FGSM Attack Simulation with Epsilon: 0.03

      ADVERSARIAL PERTURBATION METRICS      
[*] Baseline Deepfake Detection Confidence: 0.0